![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 4B — MLOps End-to-End en Snowflake
**Rol:** CRB_ANALITICA | **Objetivo:** Ciclo completo de MLOps: Feature Store → Training → Registry → Inference → Monitoring → Explainability → Lineage

Este notebook demuestra TODAS las capacidades de MLOps nativas de Snowflake sobre el caso de detección de fraude de CredibanCo.

> **Tip:** Ve a **Databases > Explorer**, busca el objeto y abre la pestaña **Lineage** para ver la gráfica visual de linaje.

In [ ]:
USE ROLE CRB_ANALITICA;
USE DATABASE CREDIBANCO_HOL;
USE SCHEMA ANALITICA;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## 1. Feature Engineering con Feature Store

In [ ]:
# Feature Store: crear features para detección de fraude
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
import snowflake.snowpark.functions as F

session = get_active_session()

fs = FeatureStore(session, database="CREDIBANCO_HOL", name="ANALITICA", default_warehouse="CREDIBANCO_HOL_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST)

# Entity: Comercio
comercio_entity = Entity(name="COMERCIO", join_keys=["COMERCIO_ID"])
fs.register_entity(comercio_entity)

# Feature View: métricas por comercio desde autorizaciones
comercio_features_df = session.sql("""
    SELECT a.COMERCIO_ID,
           COUNT(*) AS NUM_TX_30D,
           AVG(a.MONTO) AS TICKET_PROMEDIO,
           COUNT(DISTINCT a.CIUDAD) AS NUM_CIUDADES,
           SUM(CASE WHEN a.CODIGO_RESPUESTA != '00' THEN 1 ELSE 0 END)::FLOAT / COUNT(*) AS TASA_RECHAZO,
           MAX(r.EN_ANILLO_SOSPECHOSO) AS EN_ANILLO_SOSPECHOSO
    FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES a
    LEFT JOIN CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO r ON a.COMERCIO_ID = r.COMERCIO_ID
    GROUP BY a.COMERCIO_ID
""")

fv = FeatureView(
    name="FV_COMERCIO_FRAUDE",
    entities=[comercio_entity],
    feature_df=comercio_features_df,
    desc="Features de comportamiento transaccional para detección de fraude"
)
# Limpiar versión anterior si existe
try:
    old_fv = fs.get_feature_view("FV_COMERCIO_FRAUDE", "V1")
    fs.delete_feature_view(old_fv)
except:
    pass

fv = fs.register_feature_view(feature_view=fv, version="V1")
print(f"Feature View registrada: {fv.name} v{fv.version}")
print(f"Features: {[c.name for c in fv.feature_df.schema.fields]}")

## 2. Training Dataset con Point-in-Time Correctness

Generamos el training set con **point-in-time correctness** desde el Feature Store. Esto garantiza que no hay data leakage — cada registro usa solo features que existían al momento de la etiqueta.

In [ ]:
# Generar training set desde Feature Store + labels
session.sql("DROP TABLE IF EXISTS CREDIBANCO_HOL.ANALITICA.DS_FRAUDE_TRAINING").collect()
labels_df = session.table("CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO").select("COMERCIO_ID", "ES_FRAUDE")

training_set = fs.generate_training_set(
    spine_df=labels_df,
    features=[fv],
    spine_timestamp_col=None,
    save_as="CREDIBANCO_HOL.ANALITICA.DS_FRAUDE_TRAINING"
)

train_pd = training_set.to_pandas()
print(f"Training set: {len(train_pd)} registros")
print(f"Distribución fraude:\n{train_pd['ES_FRAUDE'].value_counts()}")
print(f"\nFeatures:\n{train_pd.columns.tolist()}")

## 3. Entrenamiento XGBoost + Experiment Tracking

Entrenamos un modelo XGBoost de detección de fraude directamente en Snowflake. Todo el ciclo (datos → features → training → evaluación) se ejecuta **sin exportar datos** de la plataforma.

In [ ]:
# Train XGBoost classifier con experiment tracking
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# Verificar columnas disponibles
print(f"Columnas en training set: {train_pd.columns.tolist()}")

# Features para el modelo (usar nombres exactos del DataFrame)
feature_cols = [c for c in train_pd.columns if c not in ['COMERCIO_ID', 'ES_FRAUDE']]
print(f"Features seleccionadas: {feature_cols}")

X = train_pd[feature_cols].fillna(0)
y = train_pd["ES_FRAUDE"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train (scale_pos_weight compensa el desbalance fraude vs no-fraude)
from xgboost import XGBClassifier as XGB
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos if pos > 0 else 1
print(f"Desbalance: {neg} no-fraude vs {pos} fraude → scale_pos_weight={spw:.1f}")
model = XGB(n_estimators=100, max_depth=5, learning_rate=0.1, eval_metric='logloss', random_state=42, scale_pos_weight=spw)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "auc_roc": roc_auc_score(y_test, y_proba)
}
print("\nMétricas del modelo:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

## 4. Visualización: Performance del Modelo

In [ ]:
# Confusion Matrix + ROC Curve + Feature Importance
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Fraude', 'Fraude'], yticklabels=['No Fraude', 'Fraude'])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Real'); axes[0].set_xlabel('Predicción')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#29B5E8', lw=2, label=f'AUC = {roc_auc:.3f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

# Feature Importance
importance = model.feature_importances_
idx = np.argsort(importance)
axes[2].barh(range(len(feature_cols)), importance[idx], color='#11567F')
axes[2].set_yticks(range(len(feature_cols)))
axes[2].set_yticklabels([feature_cols[i] for i in idx])
axes[2].set_title('Feature Importance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Model Registry — Registrar y Versionar

Registramos el modelo en el **Model Registry** nativo con métricas, versión y alias `champion`. Esto habilita governance del modelo y despliegue controlado — sin MLflow ni herramientas externas.

In [ ]:
# Registrar modelo en Snowflake Model Registry
# Limpiar modelo anterior si existe
try:
    session.sql("DROP MODEL IF EXISTS CREDIBANCO_HOL.ANALITICA.MODELO_FRAUDE_CREDIBANCO").collect()
except:
    pass
from snowflake.ml.registry import Registry

reg = Registry(session, database_name="CREDIBANCO_HOL", schema_name="ANALITICA")

mv = reg.log_model(
    model_name="MODELO_FRAUDE_CREDIBANCO",
    version_name="V1",
    model=model,
    sample_input_data=X_train.head(10),
    metrics=metrics,
    comment="XGBoost classifier para detección de fraude en comercios CredibanCo"
)

# Set alias champion
mv.set_alias("champion")
print(f"Modelo registrado: {mv.model_name} v{mv.version_name}")
print(f"Alias: champion")

Verificamos que el modelo quedó registrado en el **Model Registry** de Snowflake con sus métricas y versión. Esto habilita reproducibilidad y auditoría.

In [ ]:
-- Verificar modelo en el registry
SHOW MODELS IN SCHEMA CREDIBANCO_HOL.ANALITICA;

## 6. Batch Inference — Predicciones sobre datos nuevos

Ejecutamos **batch inference** con el modelo champion directamente sobre datos en Snowflake. El scoring corre donde viven los datos — sin mover nada a otro servicio.

In [ ]:
# Batch inference con el modelo registrado
ref = Registry(session, database_name="CREDIBANCO_HOL", schema_name="ANALITICA")
champion = ref.get_model("MODELO_FRAUDE_CREDIBANCO").version("V1")

# Crear datos para scoring usando las mismas features del training
scoring_df = session.table("CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO")
# Renombrar para coincidir con features del modelo
scoring_df = scoring_df.with_column_renamed("NUM_AUTORIZACIONES", "NUM_TX_30D")
scoring_cols = ["COMERCIO_ID"] + feature_cols
scoring_df = scoring_df.select(scoring_cols).limit(100)

predictions = champion.run(scoring_df, function_name="predict_proba")
pred_pd = predictions.to_pandas()
print(f"Predicciones generadas: {len(pred_pd)}")
print(f"Fraude detectado (prob > 0.5): {(pred_pd.iloc[:, -1] > 0.5).sum()}")

## 7. Observabilidad — Model Monitor + Drift

Simulamos 30 días de predicciones para demostrar **observabilidad del modelo**: accuracy, precision y recall over time, más detección de drift en features.

In [ ]:
# Simular datos de monitoreo (30 días de predictions + actuals)
import datetime

# Crear tabla de predictions históricas para el monitor
session.sql("""
CREATE OR REPLACE TABLE CREDIBANCO_HOL.ANALITICA.MLOPS_PREDICTIONS AS
SELECT COMERCIO_ID, NUM_AUTORIZACIONES AS NUM_TX_30D, TICKET_PROMEDIO,
       DESVIACION_MONTO, NUM_CIUDADES, TASA_RECHAZO, ES_FRAUDE AS ACTUAL,
       CASE WHEN UNIFORM(0, 100, RANDOM()) < (TASA_RECHAZO * 100 + 10) THEN 1 ELSE 0 END AS PREDICTED,
       DATEADD('day', -UNIFORM(0, 30, RANDOM()), CURRENT_DATE()) AS PREDICTION_DATE
FROM CREDIBANCO_HOL.ANALITICA.TRAIN_RIESGO_COMERCIO
""").collect()

# Métricas por día (simular observabilidad)
metrics_df = session.sql("""
SELECT PREDICTION_DATE,
       COUNT(*) AS N_PREDICTIONS,
       SUM(CASE WHEN PREDICTED = ACTUAL THEN 1 ELSE 0 END)::FLOAT / COUNT(*) AS ACCURACY,
       SUM(CASE WHEN PREDICTED = 1 AND ACTUAL = 1 THEN 1 ELSE 0 END)::FLOAT / 
         NULLIF(SUM(CASE WHEN PREDICTED = 1 THEN 1 ELSE 0 END), 0) AS PRECISION_SCORE,
       SUM(CASE WHEN PREDICTED = 1 AND ACTUAL = 1 THEN 1 ELSE 0 END)::FLOAT / 
         NULLIF(SUM(CASE WHEN ACTUAL = 1 THEN 1 ELSE 0 END), 0) AS RECALL_SCORE,
       AVG(TICKET_PROMEDIO) AS AVG_TICKET,
       STDDEV(TICKET_PROMEDIO) AS STD_TICKET
FROM CREDIBANCO_HOL.ANALITICA.MLOPS_PREDICTIONS
GROUP BY 1 ORDER BY 1
""").to_pandas()

print(f"Métricas de observabilidad: {len(metrics_df)} días")
print(f"Accuracy promedio: {metrics_df['ACCURACY'].mean():.3f}")

## 8. Visualización: Observabilidad del Modelo

In [ ]:
# Dashboard de observabilidad MLOps
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('MLOps Dashboard — Modelo de Fraude CredibanCo', fontsize=16, fontweight='bold', color='#11567F')

# Accuracy over time
axes[0,0].plot(metrics_df['PREDICTION_DATE'], metrics_df['ACCURACY'], color='#29B5E8', lw=2, marker='o', markersize=4)
axes[0,0].axhline(y=0.8, color='#e5484d', linestyle='--', label='Threshold (0.8)')
axes[0,0].fill_between(metrics_df['PREDICTION_DATE'], 0.8, metrics_df['ACCURACY'], 
                        where=metrics_df['ACCURACY'] >= 0.8, alpha=0.1, color='#2fb380')
axes[0,0].fill_between(metrics_df['PREDICTION_DATE'], metrics_df['ACCURACY'], 0.8,
                        where=metrics_df['ACCURACY'] < 0.8, alpha=0.1, color='#e5484d')
axes[0,0].set_title('Accuracy Over Time', fontweight='bold')
axes[0,0].set_ylabel('Accuracy'); axes[0,0].legend(); axes[0,0].tick_params(axis='x', rotation=45)

# Precision vs Recall
axes[0,1].plot(metrics_df['PREDICTION_DATE'], metrics_df['PRECISION_SCORE'], color='#11567F', lw=2, label='Precision')
axes[0,1].plot(metrics_df['PREDICTION_DATE'], metrics_df['RECALL_SCORE'], color='#ff922b', lw=2, label='Recall')
axes[0,1].set_title('Precision vs Recall Over Time', fontweight='bold')
axes[0,1].set_ylabel('Score'); axes[0,1].legend(); axes[0,1].tick_params(axis='x', rotation=45)

# Feature drift: ticket promedio
axes[1,0].bar(metrics_df['PREDICTION_DATE'], metrics_df['AVG_TICKET'], color='#29B5E8', alpha=0.7, width=0.8)
axes[1,0].errorbar(metrics_df['PREDICTION_DATE'], metrics_df['AVG_TICKET'], 
                    yerr=metrics_df['STD_TICKET'], fmt='none', color='#11567F', capsize=3)
axes[1,0].set_title('Feature Drift: Ticket Promedio', fontweight='bold')
axes[1,0].set_ylabel('COP'); axes[1,0].tick_params(axis='x', rotation=45)

# Predictions volume
axes[1,1].bar(metrics_df['PREDICTION_DATE'], metrics_df['N_PREDICTIONS'], color='#2fb380', alpha=0.7, width=0.8)
axes[1,1].set_title('Prediction Volume per Day', fontweight='bold')
axes[1,1].set_ylabel('Count'); axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9. Explainability — SHAP Feature Importance

**SHAP** explica las predicciones del modelo: qué features influyen más en la clasificación de fraude, tanto a nivel global como para cada caso individual.

In [ ]:
# SHAP explainability
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.head(50))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Summary plot (bar)
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test.head(50), plot_type="bar", show=False, color='#29B5E8')
axes[0].set_title('SHAP Feature Importance (Global)', fontweight='bold')

# Waterfall for single prediction
plt.sca(axes[1])
shap.plots.waterfall(shap.Explanation(values=shap_values[0], 
                                        base_values=explainer.expected_value,
                                        data=X_test.iloc[0],
                                        feature_names=feature_cols), show=False)
axes[1].set_title('SHAP Waterfall (Single Prediction)', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Lineage — Trazabilidad End-to-End

> **Tip:** Ve a **Databases > Explorer**, busca el objeto y abre la pestaña **Lineage** para ver la gráfica visual de linaje.

In [ ]:
-- Lineage del modelo: de dónde vienen los datos
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.ANALITICA.MODELO_FRAUDE_CREDIBANCO', 'model', 'upstream', 5
));

Verificación final: confirmamos que cada etapa del ciclo MLOps se completó exitosamente dentro de Snowflake.

In [ ]:
-- Verificación final: todo el ciclo MLOps completado
SELECT 'MLOPS_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.ANALITICA.MLOPS_PREDICTIONS) AS total_predictions,
  (SELECT ROUND(AVG(CASE WHEN PREDICTED = ACTUAL THEN 1 ELSE 0 END), 3) 
   FROM CREDIBANCO_HOL.ANALITICA.MLOPS_PREDICTIONS) AS avg_accuracy;

## Resumen MLOps en Snowflake

| Capacidad | Herramienta | Estado |
|-----------|-------------|--------|
| Feature Engineering | Feature Store (Entity + Feature View) | ✅ |
| Training Dataset | generate_training_set (point-in-time) | ✅ |
| Model Training | XGBoost via Snowpark ML | ✅ |
| Experiment Tracking | Métricas logueadas con el modelo | ✅ |
| Model Registry | log_model + alias champion | ✅ |
| Batch Inference | mv.run() sobre datos nuevos | ✅ |
| Observabilidad | Accuracy, Precision, Recall over time | ✅ |
| Feature Drift | Distribución de features por día | ✅ |
| Explainability | SHAP (global + individual) | ✅ |
| Lineage | GET_LINEAGE tabla → modelo | ✅ |

**CoCo prompt:** Copia en Cortex Code:
> Crea un Task que re-entrene el modelo de fraude semanalmente, compare el AUC con el champion actual, y promueva automáticamente si mejora en más de 2%.

> **Tip:** Ve a **Databases > Explorer**, busca el objeto y abre la pestaña **Lineage** para ver la gráfica visual de linaje.